# Preprocessing ablation for sentiment training

Goal: measure how each variant of our `TextPreprocessor` (8 variants from `sentiment_ablation_configs`) affects a BOW + logistic-regression sentiment classifier — used as a fast proxy for the eventual BERT model.

Pipeline per variant:
1. Take the same stratified sample of the corpus.
2. Preprocess every row through `TextPreprocessor(**variant_kwargs)`, dispatching the spaCy pipeline by `_lang`.
3. Stratified 80/20 train/dev split.
4. Fit `BOWLogisticRegression` (TF-IDF + bigrams + L2 — best baseline from `outputs_v2`).
5. Score on dev: MAE, competition score (`1 - MAE/4`), accuracy, macro-F1.

Sections:
- **1.** Main ablation — all 8 variants ranked.
- **2.** Dedup-impact check — does removing label-conflicting duplicates help?
- **3.** Cross-axis: best variant × different model types.
- **4.** Cross-axis: best variant × different ngram ranges.
- **Conclusion.**

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd()
if current_dir.name == "baselines":
    os.chdir("..")
project_root = str(Path.cwd().resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import spacy
from data_analysis.utils.text_analysis import (
    TextAnalyzer,
    evaluate_ablations,
    sentiment_evaluator,
)
from data_analysis.utils.text_preprocessing import TextPreprocessor
from baselines.src.utils import MetricsComputer

In [ ]:
DEBUG_PATH = "data/splitlang/DEBUG/train.csv"
SAMPLE_PER_LABEL = 2000   # 2k * 5 labels = 10k stratified sample
RANDOM_STATE = 42

debug = pd.read_csv(DEBUG_PATH)
print(f"corpus: {len(debug):,} rows   |   languages: {debug['_lang'].value_counts().to_dict()}")

sample = (
    debug.groupby("label")
         .sample(n=SAMPLE_PER_LABEL, random_state=RANDOM_STATE)
         .reset_index(drop=True)
)
print(f"sample: {len(sample):,} rows   |   per-label: {sample['label'].value_counts().sort_index().to_dict()}")
print(f"        languages: {sample['_lang'].value_counts().to_dict()}")

def _load_or_download(model_name):
    try:
        return spacy.load(model_name)
    except OSError:
        spacy.cli.download(model_name)
        return spacy.load(model_name)

nlp_en = _load_or_download("en_core_web_sm")
nlp_de = _load_or_download("de_core_news_sm")
nlp_by_lang = {"EN": nlp_en, "DE": nlp_de}
analyzer = TextAnalyzer(nlp=nlp_by_lang)

## 1. Main preprocessing ablation

All 8 variants from `sentiment_ablation_configs()`:
- `baseline` — strip emoji, cap_3 punct, strip_descriptors entities
- `emoji_keep` / `emoji_demojize` / `emoji_pos_neg_token` — emoji axis (3 alternatives)
- `punct_keep` / `punct_single` — punctuation axis (2 alternatives)
- `entity_keep` / `entity_replace_with_token` — entity axis (2 alternatives)

Each variant changes exactly one axis off the baseline. Lower MAE = better. This cell takes ~20-30 minutes on the 10k sample (most of it is spaCy NER per row).

In [ ]:
variants = analyzer.sentiment_ablation_configs()
print(f"running {len(variants)} variants on {len(sample):,} rows\n")

ablation = evaluate_ablations(
    sample_df=sample,
    variants=variants,
    text_column="sentence",
    label_column="label",
    evaluator_fn=sentiment_evaluator,
    nlp_by_lang=nlp_by_lang,
    text_lang_column="_lang",
)
ablation = ablation.sort_values("mae").reset_index(drop=True)
ablation.to_csv("baselines/outputs_v2/our_preprocessing_ablation.csv", index=False)
display(ablation[["variant", "mae", "score", "accuracy", "f1_macro", "rmse"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

baseline_idx = ablation[ablation["variant"] == "baseline"].index[0]
colors = ["#4C72B0" if i == baseline_idx else "#DD8452" for i in ablation.index]

ax = axes[0]
bars = ax.barh(ablation["variant"], ablation["mae"], color=colors, edgecolor="white")
ax.set_xlabel("validation MAE (lower is better)")
ax.set_title("BOW dev MAE per preprocessing variant")
for bar, mae in zip(bars, ablation["mae"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{mae:.4f}", va="center", fontsize=9)
ax.invert_yaxis()

ax = axes[1]
bars = ax.barh(ablation["variant"], ablation["score"], color=colors, edgecolor="white")
ax.set_xlabel("competition score = 1 - MAE/4 (higher is better)")
ax.set_title("BOW dev score per preprocessing variant")
for bar, sc in zip(bars, ablation["score"]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{sc:.4f}", va="center", fontsize=9)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig("data_analysis/img/sentiment_preprocessing_ablation.png", dpi=120, bbox_inches="tight")
plt.show()

best = ablation.iloc[0]
baseline = ablation[ablation["variant"] == "baseline"].iloc[0]
delta_mae = baseline["mae"] - best["mae"]
print(f"best variant : {best['variant']}   mae={best['mae']:.4f}   score={best['score']:.4f}")
print(f"baseline     : {baseline['variant']}   mae={baseline['mae']:.4f}   score={baseline['score']:.4f}")
print(f"gain vs baseline: {delta_mae:+.4f} MAE   ({(delta_mae / baseline['mae']) * 100:+.2f}% relative)")

## 2. Dedup impact (Build 4)

Youssef's notebook found 27 distinct preprocessed texts that appear with **multiple labels** — genuine label noise injected by aggressive normalisation. Question: does removing the rows in those conflicting groups help BOW training?

Use the **best variant from §1** to preprocess; identify groups of identical preprocessed text with ≥2 distinct labels; re-train BOW with those rows removed.

In [ ]:
best_cfg = next(c for c in variants if c["name"] == best["variant"])
best_kwargs = {k: v for k, v in best_cfg.items() if k != "name"}
best_pre = TextPreprocessor(**best_kwargs)

work = sample.copy()
work["_clean"] = [
    best_pre.preprocess(row["sentence"], nlp=nlp_by_lang.get(row["_lang"]), lang=row["_lang"].lower())
    for _, row in work.iterrows()
]

labels_per_text = work.groupby("_clean")["label"].nunique()
conflict_texts = set(labels_per_text[labels_per_text > 1].index)
conflict_mask = work["_clean"].isin(conflict_texts)
n_conflict_groups = len(conflict_texts)
n_conflict_rows = int(conflict_mask.sum())

print(f"variant used        : {best['variant']}")
print(f"distinct conflict groups: {n_conflict_groups}")
print(f"rows in conflict groups : {n_conflict_rows}  ({100 * n_conflict_rows / len(work):.2f} % of sample)")

metrics_all = sentiment_evaluator(work["_clean"].tolist(), work["label"].tolist())
metrics_dedup = sentiment_evaluator(
    work.loc[~conflict_mask, "_clean"].tolist(),
    work.loc[~conflict_mask, "label"].tolist(),
)

dedup_table = pd.DataFrame([
    {"split": "all rows",                **metrics_all},
    {"split": "conflict-rows removed",   **metrics_dedup},
])
display(dedup_table[["split", "mae", "score", "accuracy", "f1_macro"]].round(4))
delta = metrics_all["mae"] - metrics_dedup["mae"]
print(f"removing {n_conflict_rows} conflict rows changes MAE by {delta:+.4f} "
      f"({delta / metrics_all['mae'] * 100:+.2f}% relative)")

## 3. Cross-axis: model type (Build 3a)

Take the best preprocessing variant from §1 and run it through different BOW model types. If the gain transfers across model types, it is robust; if it disappears with NB/SVC, the gain is LR-specific.

In [ ]:
model_configs = [
    {"type": "bow_lr",          "name": "bow_lr",          "ngram_range": [1, 2], "max_features": 10000, "C": 1.0, "vectorizer_type": "tfidf", "solver": "lbfgs", "penalty": "l2"},
    {"type": "bow_lr_weighted", "name": "bow_lr_weighted", "ngram_range": [1, 2], "max_features": 10000, "C": 1.0, "vectorizer_type": "tfidf", "solver": "lbfgs", "penalty": "l2"},
    {"type": "bow_nb",          "name": "bow_nb",          "ngram_range": [1, 2], "max_features": 10000, "alpha": 1.0},
    {"type": "bow_svc",         "name": "bow_svc",         "ngram_range": [1, 2], "max_features": 10000, "C": 1.0, "vectorizer_type": "tfidf"},
]

rows = []
for cfg in model_configs:
    print(f"  {cfg['name']:18s} ...", end="", flush=True)
    m = sentiment_evaluator(work["_clean"].tolist(), work["label"].tolist(), model_config=cfg)
    rows.append({"model": cfg["name"], **m})
    print(f" mae={m['mae']:.4f}  score={m['score']:.4f}")

cross_model_df = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
cross_model_df.to_csv("baselines/outputs_v2/our_preprocessing_x_model_type.csv", index=False)
display(cross_model_df[["model", "mae", "score", "accuracy", "f1_macro"]].round(4))

## 4. Cross-axis: ngram range (Build 3b)

Same logic, varying the n-gram range while keeping the best preprocessing + LR fixed.

In [ ]:
ngram_configs = [
    {"type": "bow_lr", "name": "unigram",  "ngram_range": [1, 1], "max_features": 10000, "C": 1.0, "vectorizer_type": "tfidf", "solver": "lbfgs", "penalty": "l2"},
    {"type": "bow_lr", "name": "1-2 gram", "ngram_range": [1, 2], "max_features": 10000, "C": 1.0, "vectorizer_type": "tfidf", "solver": "lbfgs", "penalty": "l2"},
    {"type": "bow_lr", "name": "1-3 gram", "ngram_range": [1, 3], "max_features": 10000, "C": 1.0, "vectorizer_type": "tfidf", "solver": "lbfgs", "penalty": "l2"},
]

rows = []
for cfg in ngram_configs:
    print(f"  {cfg['name']:10s} ...", end="", flush=True)
    m = sentiment_evaluator(work["_clean"].tolist(), work["label"].tolist(), model_config=cfg)
    rows.append({"ngram": cfg["name"], **m})
    print(f" mae={m['mae']:.4f}  score={m['score']:.4f}")

cross_ngram_df = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
cross_ngram_df.to_csv("baselines/outputs_v2/our_preprocessing_x_ngram.csv", index=False)
display(cross_ngram_df[["ngram", "mae", "score", "accuracy", "f1_macro"]].round(4))

## Conclusion — what to use for retraining

After this notebook runs, fill in the table below with the actual numbers (the markdown is a template).

| question | answer |
|---|---|
| best preprocessing variant for BOW | **(fill in from §1)** |
| relative improvement vs baseline | **(fill in from §1)** |
| does removing label-conflict rows help? | **(fill in from §2)** |
| does the gain transfer across model types? | **(fill in from §3)** |
| does the gain transfer across ngram ranges? | **(fill in from §4)** |

**Decision rule for retraining (BERT or BOW final):**
- If §1 best variant is also best in §3 and §4 → use it.
- If §3 / §4 reshuffle → use the variant that wins on **§3 (LR)** since BERT decoding behaves more like LR than NB.
- If §2 dedup gives ≥ 0.01 MAE drop → run dedup as a final cleanup step on the training corpus.

All result CSVs are saved to `baselines/outputs_v2/our_preprocessing_*.csv`.